In [ ]:
import json

import pandas as pd
import numpy as np

from datasets import load_dataset
import transformers
from transformers import AutoTokenizer

In [ ]:

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("nvidia/HelpSteer3", "preference")

In [ ]:
ds['train'][0].keys()

In [ ]:
filtered = ds.filter(lambda entry: entry['overall_preference'] != 0)


In [ ]:
def to_chosen_rejected(entry):
    context = entry['context']
    if entry['overall_preference'] > 1:
        chosen = entry['response1']
        rejected = entry['response2']
    else:
        chosen = entry['response2']
        rejected = entry['response1']
    entry['chosen'] = context + [{"role": "assistant", "content": chosen}]
    entry['rejected'] = context + [{"role": "assistant", "content": rejected}]
    entry['preference_strength'] = abs(entry['overall_preference'])
    return entry

preference_ds = ds.map(to_chosen_rejected)

In [ ]:
preference_ds.push_to_hub('ktolnos/helpsteer3-preference-chosenrrejected')

In [ ]:
old_ds = load_dataset('gagan3012/helpsteer2-preference-v2')
old_ds['train'][0].keys()

In [ ]:
load_dataset('ktolnos/helpsteer3-preference-chosenrrejected')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
tokenizer.max_length = 2048

In [ ]:
chosen_messages = preference_ds['train'][0]['chosen']
chosen_messages

In [ ]:
preference_ds['train'][0]['context']

In [ ]:
tokenizer.apply_chat_template(chosen_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False, truncation=True, max_length=tokenizer.max_length)